# KhaledDSML Bayesian Analytics Studio — 0.10.0rc1

**التحليل البايزي للبيانات الثنائية | Bayesian Beta–binomial analysis**

دفتر ثنائي اللغة من إعداد مشروع KhaledDSML. الوظائف الجديدة موجودة في فرع تطوير غير منشور؛ الإصدار الرسمي على PyPI هو 0.9.0.

Bilingual research notebook for the *unreleased* v0.10.0rc1 branch. Results are computed from data, not mocked.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@main.10-catalog-henze-zirkler"

### 1. التوزيع القبلي والبعدي | Prior and posterior

اختر توزيعًا قبليًا قبل تحليل بيانات الدراسة، واحرص على بيان سبب اختياره في التقرير.
Pre-specify the Beta prior. The resulting credible interval is conditional on prior/model assumptions.

In [ ]:
import pandas as pd
from khaled_dsml import (
    beta_binomial_posterior, beta_binomial_group_comparison,
    display_analysis_dashboard,
)
posterior = beta_binomial_posterior(
    successes=18, trials=50,
    prior_alpha=1.0, prior_beta=1.0,
    credible_level=0.95, predictive_trials=12,
)
display_analysis_dashboard(posterior, theme="aurora")

### 2. فحص أثر التوزيع القبلي | Prior sensitivity

لا تُفهم النتائج بدون ملاحظة اختلاف الافتراض القبلي. مثال توضيحي لمقارنة قبلي منتظم (1,1) وجيفريز (0.5,0.5).
Investigate sensitivity to two explicit priors; do not assert that one is universally correct.

In [ ]:
results = []
for label, alpha, beta in [
    ("Uniform Beta(1,1)", 1.0, 1.0),
    ("Jeffreys Beta(0.5,0.5)", 0.5, 0.5),
]:
    r = beta_binomial_posterior(
        18, 50, prior_alpha=alpha, prior_beta=beta, predictive_trials=0
    )
    results.append({
        "prior": label,
        "posterior_mean": r["posterior_mean"],
        "credible_lower": r["credible_lower"],
        "credible_upper": r["credible_upper"],
    })
pd.DataFrame(results)

### 3. مقارنة مجموعتين مستقلتين | Two independent groups

- النموذج الصفري H0: معدل واحد مشترك
- النموذج البديل H1: معدلان مستقلان
- **BF10 ليس قيمة p ولا احتمال صحة H1**.
- The posterior probability of group superiority is conditional on H1. Do not use for repeated measures or causal claims.

In [ ]:
comparison = beta_binomial_group_comparison(
    30, 60, 19, 60,
    prior_alpha=1, prior_beta=1,
)
display_analysis_dashboard(comparison, theme="amethyst")

### 4. ملاحظات أساسية | Reporting checklist

**العربية:** اذكر عدد المحاولات والنجاحات، ومعلمات التوزيع القبلي، ونوع النموذج، ومستوى المصداقية، والافتراضات، وفروق التصميم. لا تختلق قيمًا احتمالية أو فواصل ثقة.

**English:** Report counts, Beta prior, explicit H0/H1 hypotheses, posterior credible level and model limitations. BF10 depends on the prior and is not P(H1|data).